# 열연 압연 하중 데이터 불러오기

원본: `../main/raw_data/metals.csv`

CSV 뒤에 내용 없는 빈 컬럼 18개가 붙어 있어 앞의 10개 컬럼만 읽습니다.

In [ ]:
from pathlib import Path

import pandas as pd

CSV = Path("../main/raw_data/metals.csv")

df = pd.read_csv(CSV, usecols=range(10))
print(df.shape)
df.head()

## 연속성이 깨진 코일 제외

한 코일은 F1~F5가 한 번씩 있어야 하고, 앞 스탠드 출구 두께 = 다음 스탠드 입구 두께, 폭은 코일 안에서 일정해야 합니다.
이를 어기는 코일은 행이 코일 사이에서 밀린 것으로 보고 코일 단위로 통째로 제외합니다.

In [ ]:
STANDS = ["F1", "F2", "F3", "F4", "F5"]
TOL_GAUGE = 0.01  # mm, 앞 스탠드 출구 두께와 다음 스탠드 입구 두께 허용 오차
TOL_WIDTH = 0.01  # mm, 한 코일 안에서 폭 허용 변동


def find_broken_coils(df, tol_gauge=TOL_GAUGE, tol_width=TOL_WIDTH):
    """연속성이 깨진 코일을 사유별로 찾아 {사유: 코일 ID 집합}으로 반환한다."""
    d = df.sort_values(["Coil ID", "Stand ID"])
    g = d.groupby("Coil ID")

    stand_ok = g["Stand ID"].apply(lambda s: list(s) == STANDS)
    incomplete = set(stand_ok.index[~stand_ok])

    gap = (d["Exit gauge (mm)"] - g["Entry gauge (mm)"].shift(-1)).abs()
    gauge_gap = set(d.loc[gap > tol_gauge, "Coil ID"])

    wr = g["Width (mm)"].max() - g["Width (mm)"].min()
    width_var = set(wr.index[wr > tol_width])

    return {"스탠드 구성 불완전": incomplete, "두께 불연속": gauge_gap, "폭 변동": width_var}


reasons = find_broken_coils(df)
for k, v in reasons.items():
    print(f"{k}: {len(v)}개 코일")

broken = set().union(*reasons.values())
df_clean = df[~df["Coil ID"].isin(broken)].reset_index(drop=True)
print(f"\n제외 코일 {len(broken)}개 / 전체 {df['Coil ID'].nunique()}개")
print(f"행: {len(df)} -> {len(df_clean)} ({len(df) - len(df_clean)}행 제외)")
df_clean["Grade"].value_counts().sort_index()

## 데이터 정리 기준 요약

### 1. 원본 데이터 (논문 저자가 이미 정리한 부분)
- 0 이하 값과 결측치 제거 (Grade 2의 코일 2개에서 5행)
- 단위는 전부 SI (하중 N, 길이 mm, 온도 K)

### 2. 우리가 확인만 하고 손대지 않은 것
- 두께 역전(출구 >= 입구): 0건
- 스탠드별 하중 극단값 (중앙값 ± 4.5 x IQR 밖): 0건 -> 제거하지 않음
- 결측치: 0개

### 3. 우리가 제외한 것 (코일 단위로 통째로 제외)
한 코일은 F1~F5가 한 번씩 있어야 하고, 앞 스탠드 출구 두께 = 다음 스탠드 입구 두께, 폭은 코일 안에서 일정해야 한다.
이를 어기는 코일은 행이 코일 사이에서 밀린 것으로 보고 제외한다.

| 기준 | 허용 오차 | 해당 코일 |
|---|---|---|
| 스탠드 구성이 F1~F5 한 번씩이 아님 | - | 8 |
| 앞 스탠드 출구와 다음 스탠드 입구 두께 차이 | 0.01 mm | 249 |
| 코일 내 폭 변동 | 0.01 mm | 261 |
| **합계 (중복 제거)** | | **264 (3.4%)** |

- 결과: 7,797개 -> 7,533개 코일, 38,980행 -> 37,665행 (전부 코일당 5행)
- 행 단위로 고치거나 순서를 복구하지 않음 (틀릴 위험이 있어서)
- 제외 이유: 압하 배분 최적화는 연속성(F1 출구 = F2 입구)을 전제로 하기 때문

### 4. 알려진 영향과 한계
- **Grade05**: 392개 -> 130개 코일 (262개, 67% 제외). 소수 강종이라 성능이 불안정할 수 있음
- Grade02: 1,542개 -> 1,540개. 나머지 강종은 변화 없음
- 제외된 코일은 이상한 코일이라기보다 기록이 밀린 코일이라, 제외로 인한 편향은 Grade05 외에는 작다고 본다
- 속도, 성분, 윤활 정보는 원래 데이터에 없음

### 5. 모델링 때 지킬 것
- 학습/평가는 행이 아니라 **코일 단위로** 분할 (한 코일의 5개 스탠드가 섞이지 않게)
- 강종별 성능을 따로 확인 (Grade05, Grade10 등 소수 강종)
- 최적화는 관측 범위 안으로 제한

## EDA 1. 파생 변수와 그래프 설정

- `draft`: 압하량 = 입구 두께 - 출구 두께 (mm)
- `reduction`: 압하율 = 압하량 / 입구 두께
- `strain`: 진변형률 = ln(입구 / 출구)
- `contact_len`: 접촉길이 근사 = sqrt(롤 반경 x 압하량) (mm). 물리 기준 모델의 재료
- `share`: 코일 안에서 그 스탠드가 차지하는 하중 비율 (하중 집중 지표의 재료)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Malgun Gothic")

FORCE = "Force (N)"
d = df_clean.copy()
d["draft"] = d["Entry gauge (mm)"] - d["Exit gauge (mm)"]
d["reduction"] = d["draft"] / d["Entry gauge (mm)"]
d["strain"] = np.log(d["Entry gauge (mm)"] / d["Exit gauge (mm)"])
d["contact_len"] = np.sqrt(d["Radius (mm)"] * d["draft"])
d["force_MN"] = d[FORCE] / 1e6
d["share"] = d[FORCE] / d.groupby("Coil ID")[FORCE].transform("sum")

d[["draft", "reduction", "strain", "contact_len", "force_MN", "share"]].describe().round(3)

## EDA 2. 스탠드별 하중과 압하율

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=d, x="Stand ID", y="force_MN", order=STANDS, ax=ax[0], color="#8fb7d9")
ax[0].set(title="스탠드별 압연 하중", ylabel="하중 (MN)")
sns.boxplot(data=d, x="Stand ID", y="reduction", order=STANDS, ax=ax[1], color="#e8b88a")
ax[1].set(title="스탠드별 압하율", ylabel="압하율")
sns.boxplot(data=d, x="Stand ID", y="share", order=STANDS, ax=ax[2], color="#9cc9a1")
ax[2].set(title="스탠드별 하중 비율 (코일 내)", ylabel="비율")
plt.tight_layout()
plt.show()

d.groupby("Stand ID")[["force_MN", "reduction", "share", "Temperature (K)", "Radius (mm)"]].mean().round(3)

## EDA 3. 압하율과 하중의 관계

전체 상관만 보면 스탠드 번호가 섞여 과장될 수 있으므로, **스탠드 안에서의 상관**도 같이 봅니다.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
samp = d.sample(8000, random_state=0)
sns.scatterplot(data=samp, x="reduction", y="force_MN", hue="Stand ID", hue_order=STANDS,
                s=8, alpha=0.5, ax=ax[0])
ax[0].set(title="압하율 vs 하중", xlabel="압하율", ylabel="하중 (MN)")
sns.scatterplot(data=samp, x="contact_len", y="force_MN", hue="Stand ID", hue_order=STANDS,
                s=8, alpha=0.5, ax=ax[1], legend=False)
ax[1].set(title="접촉길이 근사 vs 하중", xlabel="sqrt(R x 압하량) (mm)", ylabel="하중 (MN)")
plt.tight_layout()
plt.show()

feats = ["Entry gauge (mm)", "Exit gauge (mm)", "draft", "reduction", "strain", "contact_len",
         "Radius (mm)", "Width (mm)", "Temperature (K)"]
corr = pd.DataFrame({
    "전체": d[feats + [FORCE]].corr()[FORCE],
    **{s: d[d["Stand ID"] == s][feats + [FORCE]].corr()[FORCE] for s in STANDS},
}).drop(FORCE).round(2)
corr

## EDA 4. 강종별 하중

In [ ]:
order = sorted(d["Grade"].unique())
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [2, 1]})
sns.boxplot(data=d, x="Grade", y="force_MN", order=order, ax=ax[0], color="#8fb7d9")
ax[0].set(title="강종별 하중 (전체 스탠드)", ylabel="하중 (MN)")
ax[0].tick_params(axis="x", rotation=45)
cnt = d.groupby("Grade")["Coil ID"].nunique().reindex(order)
ax[1].bar(range(len(order)), cnt.values, color="#bbbbbb")
ax[1].set_xticks(range(len(order)))
ax[1].set_xticklabels(order, rotation=45)
ax[1].set(title="강종별 코일 수", ylabel="코일 수")
plt.tight_layout()
plt.show()

# 같은 스탠드끼리 비교해야 강종 효과가 보인다
d.pivot_table(index="Grade", columns="Stand ID", values="force_MN", aggfunc="mean").round(2)

## EDA 5. 하중 집중 (최적화의 목표 지표 후보)

코일 한 개에서 5개 스탠드 하중을 비교해 집중도를 봅니다.

- `max_share`: 가장 큰 하중이 전체에서 차지하는 비율 (균등이면 0.2)
- `cv`: 하중의 변동계수 (표준편차 / 평균)
- `max_stand`: 하중이 가장 큰 스탠드

In [ ]:
g = d.groupby("Coil ID")
coil = pd.DataFrame({
    "Grade": g["Grade"].first(),
    "max_force_MN": g["force_MN"].max(),
    "max_share": g["share"].max(),
    "cv": g["force_MN"].std(ddof=0) / g["force_MN"].mean(),
    "max_stand": d.loc[g["force_MN"].idxmax(), ["Coil ID", "Stand ID"]].set_index("Coil ID")["Stand ID"],
})

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(coil["max_share"], bins=40, ax=ax[0], color="#8fb7d9")
ax[0].axvline(0.2, color="red", ls="--", label="균등 (0.2)")
ax[0].set(title="코일별 최대 하중 비율", xlabel="max_share")
ax[0].legend()
sns.histplot(coil["max_force_MN"], bins=40, ax=ax[1], color="#e8b88a")
ax[1].set(title="코일별 최대 하중", xlabel="MN")
coil["max_stand"].value_counts().reindex(STANDS).fillna(0).plot.bar(ax=ax[2], color="#9cc9a1")
ax[2].set(title="하중이 가장 큰 스탠드", xlabel="", ylabel="코일 수")
ax[2].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

print(coil["max_stand"].value_counts(normalize=True).reindex(STANDS).fillna(0).round(3).to_dict())
coil[["max_force_MN", "max_share", "cv"]].describe().round(3)

## EDA 6. 입력 변수의 관측 범위

최적화가 학습 범위 밖으로 나가지 않도록 스탠드별 범위를 확인합니다. (외삽 방지용 경계 후보)

In [ ]:
rng = d.groupby("Stand ID")[["reduction", "draft", "Entry gauge (mm)", "Exit gauge (mm)",
                              "Temperature (K)", "Radius (mm)", "Width (mm)"]].quantile([0.01, 0.5, 0.99]).round(3)
rng.unstack(level=1)